# 02 - Bronze Auto Loader Ingestion
## 04 - Reference
### Goal
- copy raw files into a governed landing volume
- ingest files into a bronze Delta tables using Auto Loader

In [0]:
%run ../01_setup/00_config

In [0]:
dbutils.fs.mkdirs(reference_path)

for file_info in dbutils.fs.ls(f"{repo_sample_path}/reference"):
    if file_info.name.endswith(".csv"):
        try:
            dbutils.fs.cp(file_info.path, f"{reference_path}/{file_info.name}")
            print(f"Copied: {file_info.name}")
        except Exception as e:
            print(f"Copy skipped or failed: {file_info.name} - {e}")

display(dbutils.fs.ls(reference_path))


In [0]:
reference_table_map = [
    ("asset_reference",         bronze_asset_reference_table,         f"{reference_path}/asset_reference_v2.csv"),
    ("region_reference",        bronze_region_reference_table,        f"{reference_path}/region_reference_v2.csv"),
    ("market_reference",        bronze_market_reference_table,        f"{reference_path}/market_reference_v2.csv"),
    ("event_type_reference",    bronze_event_type_reference_table,    f"{reference_path}/event_type_reference_v2.csv"),
    ("weather_alert_reference", bronze_weather_alert_reference_table, f"{reference_path}/weather_alert_reference_v2.csv"),
]

for name, table, path in reference_table_map:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
        .withColumn("ingestion_ts", F.current_timestamp())
    )
    df.write.mode("overwrite").saveAsTable(table)
    print(f"Loaded: {table}")
    display(spark.table(table))